
# Stories (Theory of Mind) Prompt Experiments (Qwen3.5-0.8B)

This notebook documents **phase-wise prompt optimizations** for the **Theory of Mind** (Stories) benchmark on **Qwen3.5-0.8B**. Stories is an image multiple-choice task requiring reasoning about characters' beliefs, knowledge, and emotions.

- **37 trials** across 6 trial types: false belief (13), reality check (10), emotion reasoning (7), attribution (3), action (3), reference (1).
- **Options:** 2 (chance 50%), 3 (chance 33%), or 4 (chance 25%) image options depending on trial type.
- **Key challenge:** False belief questions require tracking what characters *know vs. don't know*.
- **Artifacts:** per-phase CSVs and logs under `results/stories-phases/`.

## Phases

| Phase | Name | Description |
|-------|------|-------------|
| 0 | Baseline | Manifest prompts + Qwen default system prompt + thinking disabled |
| 1 | Structured prompt | Story / question / options clearly separated |
| 2 | Enhanced parsing | Reverse-scan, "image/option X" patterns, last-letter fallback |
| 3 | Task-specific system prompt | "You reason about characters' beliefs and emotions" |
| 4 | False belief hint | Character knowledge tracking reminder (belief/attribution types) |
| 5 | Mental state CoT | Step-by-step: What does character KNOW? → What happened? → Answer |


In [ ]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)
RESULTS = ROOT / "results" / "prompt_optimization"prompt_optimization"/"theory-of-mind"/"internvl-3.5-4b"theory-of-mind"prompt_optimization"/"theory-of-mind"/"internvl-3.5-4b"internvl-3.5-4b"

# Load prompt builders from the experiment script
_spec = importlib.util.spec_from_file_location(
    "stories_phases", ROOT / "scripts" / "experiment_stories_phases.py"
)
sp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(sp)

manifest_rows = sp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "theory-of-mind"
TRIALS_LIST = sp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}


def build_prompt(phases: set[int], item_uid: str) -> str:
    """Reconstruct the exact prompt used for a given phase combination."""
    t = TRIALS[item_uid]
    row = t["row"]
    pp = t["prompt_phrase"]
    n_opts = t["n_options"]
    if 1 in phases:
        p = sp.build_prompt_phase1(row, pp, n_opts)
    else:
        p = sp.build_prompt_baseline(row, pp)
    if 4 in phases:
        p = sp.apply_phase4_belief_hint(p, t["trial_type"])
    if 5 in phases:
        p = sp.apply_phase5_cot(p, t["trial_type"])
    if 3 in phases:
        p = sp.apply_phase3_system(p)
    return p


def load_result(csv_name: str, item_uid: str) -> dict | None:
    path = RESULTS / csv_name
    if not path.exists():
        return None
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["item_uid"] == item_uid:
                return row
    return None


## Summary: All Runs

Aggregate accuracy, parse rate, and delta vs baseline across all phase configurations.

In [ ]:
PHASE_CSVS = [
    ("phase_baseline.csv", "0 — Baseline"),
    ("phase_1.csv", "1 — Structured prompt"),
    ("phase_2.csv", "2 — Enhanced parsing"),
    ("phase_3.csv", "3 — ToM system prompt"),
    ("phase_4.csv", "4 — False belief hint"),
    ("phase_5.csv", "5 — Mental state CoT (512 tok)"),
    ("phase_6.csv", "6 — Answer-last CoT (512 tok)"),
    ("phase_7.csv", "7 — Type-specific system prompt"),
    ("phase_8.csv", "8 — Perspective anchor hint"),
    ("phase_1_2_3.csv", "1+2+3 — Structural"),
    ("phase_1_4.csv", "1+4 — Structured + belief hint"),
    ("phase_1_2_3_4_5.csv", "1+2+3+4+5 — All"),
    ("phase_1_2_3_5.csv", "1+2+3+5 — Structural + CoT"),
    ("phase_1_2_3_6.csv", "1+2+3+6 — Structural + answer-last CoT"),
    ("phase_3_7.csv", "3+7 — Generic + type-specific system prompts"),
    ("phase_3_8.csv", "3+8 — Generic system + perspective anchor"),
    ("phase_7_8.csv", "7+8 — Type-specific system + perspective anchor"),
    ("phase_3_7_8.csv", "3+7+8 — All system prompt variants"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if acc == baseline_acc and label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
print(df_summary.to_string(index=False))
df_summary.style.hide(axis="index")


                                          Phase  N Accuracy Parse % Δ vs baseline
                                   0 — Baseline 37    45.9%  100.0%             —
                          1 — Structured prompt 37    48.6%  100.0%       +2.7 pp
                           2 — Enhanced parsing 37    45.9%  100.0%       +0.0 pp
                          3 — ToM system prompt 37    56.8%  100.0%      +10.8 pp
                          4 — False belief hint 37    40.5%  100.0%       -5.4 pp
                 5 — Mental state CoT (512 tok) 37    43.2%  100.0%       -2.7 pp
                  6 — Answer-last CoT (512 tok) 37    43.2%  100.0%       -2.7 pp
                7 — Type-specific system prompt 37    43.2%  100.0%       -2.7 pp
                    8 — Perspective anchor hint 37    45.9%  100.0%       +0.0 pp
                             1+2+3 — Structural 37    54.1%  100.0%       +8.1 pp
                 1+4 — Structured + belief hint 37    51.4%  100.0%       +5.4 pp
                

Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline,37,45.9%,100.0%,—
1 — Structured prompt,37,48.6%,100.0%,+2.7 pp
2 — Enhanced parsing,37,45.9%,100.0%,+0.0 pp
3 — ToM system prompt,37,56.8%,100.0%,+10.8 pp
4 — False belief hint,37,40.5%,100.0%,-5.4 pp
5 — Mental state CoT (512 tok),37,43.2%,100.0%,-2.7 pp
6 — Answer-last CoT (512 tok),37,43.2%,100.0%,-2.7 pp
7 — Type-specific system prompt,37,43.2%,100.0%,-2.7 pp
8 — Perspective anchor hint,37,45.9%,100.0%,+0.0 pp
1+2+3 — Structural,37,54.1%,100.0%,+8.1 pp


## Per-Trial-Type Breakdown

Accuracy by trial type for each phase. Especially interesting:
- **false_belief_question**: the hardest type, requiring theory of mind
- **emotion_reasoning_question**: 4-option, mapping situations to emotions
- **reality_check_question**: easiest, asks about objective facts

In [ ]:
def breakdown_by_trial_type(csv_name: str, label: str):
    path = RESULTS / csv_name
    if not path.exists():
        print(f"  {csv_name} not found — skipped")
        return
    with open(path, newline="", encoding="utf-8") as f:
        data = list(csv.DictReader(f))
    by_type: dict[str, dict] = {}
    for r in data:
        tt = r["trial_type"]
        rec = by_type.setdefault(tt, {"N": 0, "Correct": 0, "Parsed": 0})
        rec["N"] += 1
        rec["Correct"] += int(r["is_correct"].lower() in ("true", "1"))
        rec["Parsed"] += int(r["parsed"].lower() in ("true", "1"))
    rows = []
    for tt, rec in sorted(by_type.items(), key=lambda kv: kv[1]["Correct"] / max(kv[1]["N"], 1), reverse=True):
        n = rec["N"]
        rows.append({"Trial Type": tt, "N": n, "Accuracy": f"{rec['Correct']/n:.0%}", "Parse %": f"{rec['Parsed']/n:.0%}"})
    total_n = sum(rec["N"] for rec in by_type.values())
    total_c = sum(rec["Correct"] for rec in by_type.values())
    total_p = sum(rec["Parsed"] for rec in by_type.values())
    rows.append({"Trial Type": "OVERALL", "N": total_n, "Accuracy": f"{total_c/total_n:.0%}", "Parse %": f"{total_p/total_n:.0%}"})
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))


for csv_name, label in PHASE_CSVS:
    breakdown_by_trial_type(csv_name, label)


  0 — Baseline
                Trial Type  N Accuracy Parse %
emotion_reasoning_question  7      71%    100%
     false_belief_question 13      54%    100%
    reality_check_question 10      40%    100%
      attribution_question  3      33%    100%
        reference_question  1       0%    100%
           action_question  3       0%    100%
                   OVERALL 37      46%    100%

  1 — Structured prompt
                Trial Type  N Accuracy Parse %
        reference_question  1     100%    100%
emotion_reasoning_question  7      71%    100%
    reality_check_question 10      50%    100%
     false_belief_question 13      38%    100%
      attribution_question  3      33%    100%
           action_question  3      33%    100%
                   OVERALL 37      49%    100%

  2 — Enhanced parsing
                Trial Type  N Accuracy Parse %
emotion_reasoning_question  7      71%    100%
     false_belief_question 13      54%    100%
    reality_check_question 10      40%    

---

## Phase 0 — Baseline

Uses the manifest prompt: a dense single-line narrative that includes the full story, question, and image references.

**Prompt format (abbreviated):**
```
Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book. Where will Madison look for her book first? Answer A or B. A: <image1>; B: <image2>
```

**Note:** Qwen3.5 thinking mode is disabled via `enable_thinking=False`.

In [ ]:
# Phase 0: false belief question
uid0_fb = "tom_reality_known_false_belief"
print("=== PROMPT (Phase 0, false belief) ===")
print(build_prompt(set(), uid0_fb))
r = load_result("phase_baseline.csv", uid0_fb)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet — run the experiment first)")

=== PROMPT (Phase 0, false belief) ===
Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book. Where will Madison look for her book first? Answer A or B. A: <image1>; B: <image2>

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


In [ ]:
# Phase 0: emotion reasoning question
uid0_em = "tom_moral_reasoning_emotion_reasoning_1"
print("=== PROMPT (Phase 0, emotion reasoning) ===")
print(build_prompt(set(), uid0_em))
r = load_result("phase_baseline.csv", uid0_em)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 0, emotion reasoning) ===
This morning when he came to school, Ethan put his book on the shelf above the coat hooks. Ethan's book is blue! And then he went outside to play. Here's Ethan playing outside. And while he was outside playing, Ethan's book fell down from above the coat hooks onto the floor under the coats. Then Hannah comes in and Hannah puts her book up above the coat hooks. Hannah's book is also blue! Ethan is still outside playing, so he didn't see what Hannah was doing. Let's see what he does. Oh look, here is Ethan reaching for this book up on the shelf. And now Hannah sees Ethan reaching for that book... the book that is Hannah's! How do you think Hannah feels about Ethan reaching for that book? Answer A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT ===
Response: C
Predicted: C  Correct: C  ✓


---

## Phase 1 — Structured Prompt

Separates the dense manifest prompt into three clear blocks:

```
Story:
Here is Madison. This morning, Madison put her book behind the chair...

Question: Where will Madison look for her book first?

A: <image1>
B: <image2>

Answer with one letter.
```

**Rationale:** Stories are very long (100–200 words). Separating the narrative from the question and options helps the model locate the key question without re-reading the entire prompt.

In [ ]:
uid1 = "tom_reality_known_false_belief"
print("=== PROMPT (Phase 1) ===")
print(build_prompt({1}, uid1))
r = load_result("phase_1.csv", uid1)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 1) ===
Story:
Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book.

Question: Where will Madison look for her book first?

A: <image1>
B: <image2>

Answer with one letter.

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


---

## Phase 2 — Enhanced Parsing

Same prompt as baseline, but with robust fallback parsing:
- `"Image A"`, `"Option A"`, `"Picture A"` patterns
- `"I choose/select A"`, `(A)` extraction
- Reverse sentence scan, last standalone letter A–D

**Rationale:** With image options, Qwen3.5 may respond with phrases like "I think the answer is Image B" rather than just "B".

In [ ]:
uid2 = "tom_interpretation_false_belief"
print("=== PROMPT (Phase 2 — same prompt, parsing differs) ===")
print(build_prompt(set(), uid2))
r_base = load_result("phase_baseline.csv", uid2)
r_p2 = load_result("phase_2.csv", uid2)
if r_base:
    print(f"\n--- Baseline parse ---")
    print(f"  Response: {r_base['raw_response']}")
    print(f"  Predicted: {r_base['predicted_label']}  Parsed: {r_base['parsed']}")
if r_p2:
    print(f"\n--- Phase 2 parse ---")
    print(f"  Response: {r_p2['raw_response']}")
    print(f"  Predicted: {r_p2['predicted_label']}  Parsed: {r_p2['parsed']}")

=== PROMPT (Phase 2 — same prompt, parsing differs) ===
Joshua, Isabel, and Ivan are playing a game. Here's how the game goes: One kid hides, but draws a clue in the sandbox so the other kids can find them. So Ivan is going first. He draws this clue in the sandbox, and then he goes to hide. Now it's Joshua's turn to leave a clue, and hide. Isabel and Ivan will then try to use the clue to find him. Joshua decides to hide behind the fountain. He draws a clue in the sand. He tries to draw the fountain, with the bottom part and the water coming out of it. But the drawing looks like a tree instead! Then Joshua goes and hides behind the fountain. When Isabel and Ivan see this clue, where will they look first for Joshua? Answer A or B. A: <image1>; B: <image2>

--- Baseline parse ---
  Response: B
  Predicted: B  Parsed: True

--- Phase 2 parse ---
  Response: B
  Predicted: B  Parsed: True


---

## Phase 3 — Task-Specific System Prompt

Replaces the generic Qwen system prompt with:
- **System:** `"You are reasoning about characters in a story. Characters may have false beliefs about events they did not witness. Always respond with exactly one letter."`
- **Suffix:** `"Respond with exactly one letter. Nothing else."`

**Rationale:** Priming the model for Theory of Mind reasoning may help it adopt the perspective-taking mindset needed for false belief questions.

In [ ]:
uid3 = "tom_moral_reasoning_false_belief"
print("=== PROMPT (Phase 3) ===")
print(build_prompt({3}, uid3))
r = load_result("phase_3.csv", uid3)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 3) ===
This morning when he came to school, Ethan put his book on the shelf above the coat hooks. Ethan's book is blue! And then he went outside to play. Here's Ethan playing outside. And while he was outside playing, Ethan's book fell down from above the coat hooks onto the floor under the coats. Then Hannah comes in and Hannah puts her book up above the coat hooks. Hannah's book is also blue! Ethan is still outside playing, so he didn't see what Hannah was doing. Now, when Ethan comes in from outside, where will he look first for his book? Answer A or B. A: <image1>; B: <image2>

Respond with exactly one letter. Nothing else.

=== MODEL OUTPUT ===
Response: A
Predicted: A  Correct: A  ✓


---

## Phase 4 — False Belief Hint

Prepends a knowledge-tracking reminder for false belief and attribution questions:

```
Important: A character only knows what they have personally seen or been told.
Events that happened while they were away are unknown to them.
```

For emotion reasoning questions:
```
Think about how the situation would make the character feel,
based on what they know and experience.
```

**Rationale:** The classic false belief error is assuming the character knows what the reader knows. This hint directly addresses that pitfall.

In [ ]:
uid4 = "tom_reality_known_false_belief"
print("=== PROMPT (Phase 4, false belief) ===")
print(build_prompt({4}, uid4))
print()

uid4_em = "tom_deception_emotion_reasoning_1"
print("=== PROMPT (Phase 4, emotion reasoning) ===")
print(build_prompt({4}, uid4_em))

r = load_result("phase_4.csv", uid4)
if r:
    print(f"\n=== MODEL OUTPUT (false belief) ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 4, false belief) ===
Important: A character only knows what they have personally seen or been told. Events that happened while they were away are unknown to them.

Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book. Where will Madison look for her book first? Answer A or B. A: <image1>; B: <image2>

=== PROMPT (Phase 4, emotion reasoning) ===
Think about how the situation would make the character feel, based on what they know and experience.

Joshua brought his toy truck with him to the park, but now he can't find it. He's looking everywhere but he can't find it. How does Joshua feel about losing his truck? Answer A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>

=== MODEL OUTPUT (false belief) ===
Response: B
Predicted: B  Correct: B  ✓


---

## Phase 5 — Mental State CoT

Appends step-by-step reasoning instructions tailored to the question type:

**For false belief / attribution:**
```
Step 1: What does this character KNOW (what did they see)?
Step 2: What actually happened (the reality)?
Step 3: Based on their knowledge (not reality), what would they think or do?
State your final answer as a single letter.
```

**For emotion reasoning:**
```
Step 1: What happened in the story?
Step 2: How does this situation affect the character?
Step 3: Which emotion best matches their experience?
State your final answer as a single letter.
```

**Rationale:** Explicit decomposition may help the model separate character knowledge from reader knowledge, which is the core skill tested by false belief tasks.

In [ ]:
uid5 = "tom_reality_known_false_belief"
print("=== PROMPT (Phase 5, false belief) ===")
print(build_prompt({5}, uid5))
r = load_result("phase_5.csv", uid5)
if r:
    print(f"\n=== MODEL OUTPUT ===")
    print(f"Response: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== PROMPT (Phase 5, false belief) ===
Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book. Where will Madison look for her book first? Answer A or B. A: <image1>; B: <image2>

Step 1: What does this character KNOW (what did they see)?
Step 2: What actually happened (the reality)?
Step 3: Based on their knowledge (not reality), what would they think or do?
State your final answer as a single letter.

=== MODEL OUTPUT ===
Response: B
Predicted: B  Correct: B  ✓


---

## Combined Runs

### Phase 1+2+3 — Structural Improvements

Structured prompt + enhanced parsing + ToM system prompt.

In [ ]:
uid_c1 = "tom_moral_reasoning_false_belief"
print("=== Phase 1+2+3 ===")
print(build_prompt({1, 2, 3}, uid_c1))
r = load_result("phase_1_2_3.csv", uid_c1)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+2+3 ===
Story:
This morning when he came to school, Ethan put his book on the shelf above the coat hooks. Ethan's book is blue! And then he went outside to play. Here's Ethan playing outside. And while he was outside playing, Ethan's book fell down from above the coat hooks onto the floor under the coats. Then Hannah comes in and Hannah puts her book up above the coat hooks. Hannah's book is also blue! Ethan is still outside playing, so he didn't see what Hannah was doing.

Question: Now, when Ethan comes in from outside, where will he look first for his book?

A: <image1>
B: <image2>

Answer with one letter.

Respond with exactly one letter. Nothing else.

Response: A
Predicted: A  Correct: A  ✓


### Phase 1+4 — Structured + Belief Hint

Structured prompt with the false belief / emotion knowledge hint.

In [ ]:
uid_c2 = "tom_interpretation_false_belief"
print("=== Phase 1+4 ===")
print(build_prompt({1, 4}, uid_c2))
r = load_result("phase_1_4.csv", uid_c2)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+4 ===
Important: A character only knows what they have personally seen or been told. Events that happened while they were away are unknown to them.

Story:
Joshua, Isabel, and Ivan are playing a game. Here's how the game goes: One kid hides, but draws a clue in the sandbox so the other kids can find them. So Ivan is going first. He draws this clue in the sandbox, and then he goes to hide. Now it's Joshua's turn to leave a clue, and hide. Isabel and Ivan will then try to use the clue to find him. Joshua decides to hide behind the fountain. He draws a clue in the sand. He tries to draw the fountain, with the bottom part and the water coming out of it. But the drawing looks like a tree instead! Then Joshua goes and hides behind the fountain.

Question: When Isabel and Ivan see this clue, where will they look first for Joshua?

A: <image1>
B: <image2>

Answer with one letter.

Response: B
Predicted: B  Correct: A  ✗


### Phase 1+2+3+4+5 — All Improvements

Full pipeline: structured prompt + enhanced parsing + ToM system prompt + belief hint + mental state CoT.

In [ ]:
uid_all = "tom_reality_known_false_belief"
print("=== Phase 1+2+3+4+5 ===")
print(build_prompt({1, 2, 3, 4, 5}, uid_all))
r = load_result("phase_1_2_3_4_5.csv", uid_all)
if r:
    print(f"\nResponse: {r['raw_response']}")
    print(f"Predicted: {r['predicted_label']}  Correct: {r['correct_label']}  {'✓' if r['is_correct']=='True' else '✗'}")
else:
    print("  (no results yet)")

=== Phase 1+2+3+4+5 ===
Important: A character only knows what they have personally seen or been told. Events that happened while they were away are unknown to them.

Story:
Here is Madison. This morning, Madison put her book behind the chair, because she didn't want anyone to find it. But when Madison was outside playing, someone did find it! And hid it under the rug. So now it's reading time and Madison wants her book.

Question: Where will Madison look for her book first?

A: <image1>
B: <image2>

Answer with one letter.

Step 1: What does this character KNOW (what did they see)?
Step 2: What actually happened (the reality)?
Step 3: Based on their knowledge (not reality), what would they think or do?
State your final answer as a single letter.

Respond with exactly one letter. Nothing else.

Response: B
Predicted: B  Correct: B  ✓


---

## Conclusions — Stories (Theory of Mind Task)

### Task description

Theory of Mind measures the ability to reason about characters' mental states, beliefs, and emotions using multi-image story sequences. Each trial presents a short narrative and asks a question about what a character knows, feels, or does. There are 37 trials across 6 question types: false belief, reality check, emotion reasoning, attribution, action, and reference questions. The model must answer with a single letter. Chance level is approximately 50% for binary trials and 33% for 3-choice trials.

### Results summary (18 configurations)

| Phase | Accuracy | Parse % | Δ |
|-------|----------|---------|---|
| 0 — Baseline | 45.9% | 100% | — |
| 1 — Structured prompt | 48.6% | 100% | +2.7 pp |
| 2 — Enhanced parsing | 45.9% | 100% | +0.0 pp |
| 3 — ToM system prompt | 56.8% | 100% | +10.8 pp |
| 4 — False belief hint | 40.5% | 100% | −5.4 pp |
| 5 — Mental state CoT (512 tok) | 43.2% | 100% | −2.7 pp |
| 6 — Answer-last CoT (512 tok) | 43.2% | 100% | −2.7 pp |
| 7 — Type-specific system prompt | 43.2% | 100% | −2.7 pp |
| 8 — Perspective anchor hint | 45.9% | 100% | +0.0 pp |
| 1+2+3 — Structural | 54.1% | 100% | +8.1 pp |
| 1+4 — Structured + belief hint | 51.4% | 100% | +5.4 pp |
| 1+2+3+4+5 — All | 54.1% | 100% | +8.1 pp |
| 1+2+3+5 — Structural + CoT | 54.1% | 100% | +8.1 pp |
| 1+2+3+6 — Structural + answer-last CoT | 54.1% | 100% | +8.1 pp |
| 3+7 — Generic + type-specific system prompts | 54.1% | 100% | +8.1 pp |
| **3+8 — Generic system + perspective anchor** | **59.5%** | **100%** | **+13.5 pp** |
| 7+8 — Type-specific system + perspective anchor | 56.8% | 100% | +10.8 pp |
| 3+7+8 — All system prompt variants | 56.8% | 100% | +10.8 pp |

### Key findings

**1. Phase 3+8 is the best configuration overall (+13.5 pp, 45.9% → 59.5%)**
Combining the generic ToM system prompt (Phase 3) with a per-turn perspective anchor hint (Phase 8) achieves the highest accuracy in the entire experiment. This suggests that two complementary mechanisms work best together: (a) establishing the reasoning frame at system level, and (b) anchoring the character's epistemic state within each user turn.

**2. Phase 3 alone is the best single-phase improvement (+10.8 pp, 45.9% → 56.8%)**
Framing the model as a Theory of Mind reasoner — telling it that characters may have false beliefs about events they did not witness — produces the largest individual gain. This outperforms all multi-phase structural combinations.

**3. Phase 8 (perspective anchor) is neutral alone but powerful in combination**
By itself, Phase 8 yields no improvement (+0.0 pp). However, paired with Phase 3, it adds +2.7 pp on top of Phase 3's +10.8 pp gain. This interaction suggests that the perspective hint only helps once the model has already been primed to reason about mental states via the system prompt.

**4. Phase 7 (type-specific system prompts) hurts alone and slightly limits Phase 3 when combined**
Phase 7 alone degrades accuracy to 43.2% (−2.7 pp). When combined with Phase 3 (3+7), the result is only 54.1%, lower than Phase 3 alone (56.8%). The type-specific instructions appear to override or dilute the more effective generic ToM framing.

**5. The structural combination (1+2+3) reaches +8.1 pp but does not beat Phase 3 alone**
Separating the prompt into Story / Question / Options sections, adding robust parsing, and applying the ToM system prompt together reach 54.1% — lower than Phase 3 alone (56.8%). Phase 1's restructuring slightly interferes with the system prompt's effect.

**6. Phase 4 (false belief hint) hurts across the board (−5.4 pp)**
An explicit per-turn reminder about false beliefs consistently degrades performance. Repeating the false belief instruction in the user turn appears to conflict with the model's internal reasoning.

**7. Phases 5 and 6 (CoT variants) both hurt or are neutral (−2.7 pp)**
Step-by-step reasoning instructions, even with answer-last formatting (Phase 6), do not improve accuracy. For Theory of Mind at 4B scale, the bottleneck is reasoning capacity, not output format.

**8. Parse rate is 100% across all 18 configurations**
InternVL3.5-4B produces consistently parseable outputs in all configurations.

### Production recommendation

Use **Phase 3+8** for Stories / Theory of Mind with InternVL3.5-4B:
- System prompt: *"You are reasoning about characters in a story. Characters may have false beliefs about events they did not witness. Always respond with exactly one letter."*
- Per-turn anchor: append a one-line note anchoring the character's perspective (e.g., "Note: [Name] only knows what they saw before leaving. Answer from [Name]'s perspective.")
- Keep the baseline manifest prompt unchanged in the user turn
- Set `max_new_tokens = 128`

If per-turn character name extraction is not practical, **Phase 3 alone** is a simpler and nearly as effective alternative (+10.8 pp).

### Limitations

- Only 37 trials — differences of ≤2 items (≤5 pp) may reflect noise rather than a real effect
- CoT was tested at 512 tokens; fully unconstrained generation might yield different results
- Phase 3+8 was not yet tested in combination with the structural phases (1+2+3+8); this could push accuracy further
- A direct comparison between InternVL3.5-4B and InternVL3.5-8B on this task was not conducted
